# Ensemble

**Domain:** Symbolic AI & Logic  ·  **from study list**  ·  **runnable:** no — conceptual / CLI / snippets  ·  _niche/proprietary DSL_

> A self-contained refresher on the **Ensemble Engine**: the open-source, JavaScript "social physics" engine — the spiritual successor to *Comme il Faut* (CiF), the system behind *Prom Week*. This notebook is the **engine-specific** companion to [`social-physics.ipynb`](social-physics.ipynb), which covers the broader idea. Here we get concrete: Ensemble's data files, its rule/volition/action model, and its actual JS API.

## 1. What & Why

**What it is.** Ensemble is a **rules-based AI framework for social simulation** authored at UC Santa Cruz's Expressive Intelligence Studio. It is a small JavaScript library plus a desktop **Authoring Tool**. You feed it data — a *schema* of what social facts can exist, a *cast* of characters, *trigger rules*, *volition rules*, and an *action library* — and it computes, every step, **who wants to do what to whom** and how the social world changes when they do it. It is the open, documented re-implementation of *Comme il Faut* (the C# engine behind *Prom Week*, 2012), with the scoring/rule core generalized into a reusable "**scored rule engine**".

**The problem it solves.** Hand-authoring reactive social NPCs (dialogue trees, scripted relationships) explodes combinatorially and shatters the moment a player does something unplanned. Ensemble lets you instead author the *rules of a social world* — "shy people hold back", "you flirt with those you like", "betrayal makes enemies" — and let the drama **emerge**. It gives you a *queryable social database*, a *scored rule engine* over it, and an *action selection* loop, as a dependency you can drop into a game.

**Reach for it when:** you want systemic, emergent relationships in a game/sim and you'd rather author ~hundreds of small social rules than thousands of story branches; you want the result to be **explainable** (every volition score decomposes into named rules) and **deterministic** (unlike an LLM NPC).

**Skip it when:** you need tight authorial control of exact story beats (use branching/storylets), the cast is huge and performance-critical, or social state is purely cosmetic. See §7 for the honest trade-offs and [`social-physics.ipynb`](social-physics.ipynb) §1 for the wider lineage (CiF → Ensemble → Versu).

## 2. Mental Model

**Ensemble = a tiny social database + a scored rule engine + an action picker.** Think of it as Datalog/Prolog facts (see [`datalog.ipynb`](datalog.ipynb)) with *utility scoring* bolted on top.

```
        ┌──────────────────────────────────────────────┐
        │  SOCIAL RECORD  (the DB of timestamped facts)  │
        │  traits · networks · relationships · status    │
        └───────────────┬────────────────────────────────┘
                        │ predicates query/mutate it
        ┌───────────────┴───────────────┐
        │                               │
   TRIGGER RULES                   VOLITION RULES
   keep state consistent           score *desires* (intents)
   "dating ⇒ not single"           "like(A,B) & shy(A) ⇒ +3 askOut"
        │                               │
        └───────────────┬───────────────┘
                        ▼
   calculateVolition(cast)  →  ranked intents per character pair
                        ▼
   getAction(A, B, volitions, cast)  →  best matching ACTION
                        ▼
   doAction(boundAction)  →  apply effects  →  social record changes
                        ▼
   setupNextTimeStep()  →  runTriggerRules()  →  loop
```

The slogan from the social-physics framing holds: **forces → integrate → new positions** becomes **rules → sum weights → new social state**. Ensemble's specific contribution is factoring that into *two* rule kinds (triggers for *truth-maintenance*, volitions for *desire*) and a separate, reusable **action library** that turns a desired *intent* into a concrete, effectful social move.

## 3. Key Concepts

| Term | What it means in Ensemble |
| --- | --- |
| **Social record** | The timestamped database of every social fact. `set`/`get` read & write it; `dumpSocialRecord()` prints it. Facts persist across time steps so rules can reference history. |
| **Schema** (a.k.a. base blueprints) | The *vocabulary*: a list of **categories** (e.g. `trait`, `network`, `relationship`), each with `isBoolean`, `directionType` (`directed` / `undirected` / `reciprocal`), `defaultValue`, optional `min`/`max`, and the `types` it contains. Loaded via `loadSocialStructure`. |
| **Predicate** | One social fact or a query/condition over it: `{category, type, first, second, value, operator}`. `first`/`second` are characters (`second` omitted for undirected facts); `operator` (`>`,`<`,`=`) + `value` make it a condition. |
| **Cast / characters** | The agents (`addCharacters`). Most social state is **relational** — defined *between* a `first` and `second`, not just within one character. |
| **Trigger rule** | Truth-maintenance: `{conditions:[...], effects:[...]}`. After any change, `runTriggerRules(cast)` fires these to keep the record consistent ("if A & B are dating, neither is single"). |
| **Volition rule** | Desire: conditions → a **weighted** push toward an **intent** (a social fact a character wants to make true). Summed by `calculateVolition` to score how much A wants each possible change toward B. |
| **Intent** | The *goal* of a volition — "A wants `romance(A,B)` to go up". Actions are matched to intents. |
| **Action** | The concrete, performable move: `{name, intent, conditions, influenceRules, effects, isAccept, leadsTo}`. `effects` mutate the record; `influenceRules` fine-tune its score; `leadsTo` chains follow-up actions (accept/reject branches). |
| **Volition** | The computed, *relative* desire of an initiator (and responder) for an action — the summed weights. Drives both *selection* (what happens) and *outcome* (accept vs reject). |
| **Time step** | `setupNextTimeStep()` advances the clock; the social record is sliced by time so rules can ask "did X happen recently?" |

## 4. Setup

**This is not Python-runnable.** Ensemble is a JavaScript library + authored data + a desktop authoring tool — not a `pip` package you drive from a notebook cell. So everything below is **real CLI commands and config/code snippets**, not `print()` theater.

```bash
# Get the engine (open source, JavaScript)
git clone https://github.com/ensemble-engine/ensemble.git
cd ensemble

# It ships as a browser/Node library. Smoke-test from Node that it loads:
node -e "const ensemble = require('./js/ensemble/ensemble.js'); \
         console.log('loaded:', typeof ensemble.init, typeof ensemble.calculateVolition);"
```

Typical integration in a web game (browser):

```html
<script src="js/ensemble/ensemble.js"></script>
<script>
  ensemble.init();                          // initialize the engine
  ensemble.loadSocialStructure(schema);     // your schema.json (object or JSON string)
  ensemble.addCharacters(castJSON);         // cast.json
  ensemble.addRules(triggerRulesJSON);      // type:"trigger"
  ensemble.addRules(volitionRulesJSON);     // type:"volition"
  ensemble.addActions(actionsJSON);         // the action library
  ensemble.addHistory(historyJSON);         // optional seed of the social record
</script>
```

**Authoring is data-first.** You spend most of your time editing five JSON-ish files — `schema`, `cast`, `triggerRules`, `volitionRules`, `actions` — and a `history` seed. The **Ensemble Authoring Tool** (a separate desktop app, also on the ensemble-engine GitHub org) exists specifically to edit *and debug* these rule sets, because reasoning about emergent rule interactions by hand is the hard part (see §6).

## 5. Worked Examples

Conceptual walkthroughs in **Ensemble-flavored JSON / pseudo-code — not executed** (Ensemble runs in JS, not this kernel). Field names follow Ensemble's data model; treat exact spellings as schematic and check the wiki for your version.

### Example 1 — Schema, cast, and the social record

The schema declares what *kinds* of facts can exist; the cast names the agents; then you seed the record.

```jsonc
// schema.json — the vocabulary of the social world
[
  { "category": "trait",        "isBoolean": true,  "directionType": "undirected",
    "types": ["shy", "confident", "popular"] },
  { "category": "network",      "isBoolean": false, "directionType": "directed",
    "min": 0, "max": 100,
    "types": ["romance"] },                  // how much `first` is into `second`
  { "category": "relationship", "isBoolean": true,  "directionType": "reciprocal",
    "types": ["dating", "friends", "enemies"] }
]
```

```jsonc
// cast.json
{ "cast": ["Alice", "Bob", "Carol"] }
```

```js
// seed the social record (or load via addHistory)
ensemble.set({category:"trait",   type:"shy",     first:"Alice", value:true});
ensemble.set({category:"network", type:"romance", first:"Alice", second:"Bob", value:70});
ensemble.set({category:"network", type:"romance", first:"Bob",   second:"Alice", value:20});

ensemble.get({category:"network", type:"romance", first:"Alice", second:"Bob"});
// -> [{... value:70 ...}]   (get returns matching social-record entries)
```

### Example 2 — Volition rules, trigger rules, and an action

**Volition rules** score *desire* toward an intent; **trigger rules** keep the record consistent; the **action** carries the effects.

```jsonc
// volitionRules.json  — "how much does `first` want to date `second`?"
{ "fileName": "romanceVolitions", "type": "volition", "rules": [
  { "name": "into them -> want to date",
    "conditions": [
      {"category":"network","type":"romance","first":"x","second":"y","operator":">","value":60} ],
    "effects": [   // a positive push toward the dating intent, weighted
      {"category":"relationship","type":"dating","first":"x","second":"y","value":true,"weight":5} ] },

  { "name": "shy -> hold back",
    "conditions": [ {"category":"trait","type":"shy","first":"x","value":true} ],
    "effects":    [ {"category":"relationship","type":"dating","first":"x","second":"y","value":true,"weight":-4} ] }
]}
```

```jsonc
// triggerRules.json  — truth maintenance, no weights
{ "fileName": "consistency", "type": "trigger", "rules": [
  { "name": "dating implies not single",
    "conditions": [ {"category":"relationship","type":"dating","first":"x","second":"y","value":true} ],
    "effects":    [ {"category":"trait","type":"single","first":"x","value":false} ] }
]}
```

```jsonc
// actions.json  — the performable move tied to the dating intent
{ "actions": [
  { "name": "askOut",
    "intent": {"category":"relationship","type":"dating","first":"initiator","second":"responder","value":true},
    "conditions": [ {"category":"relationship","type":"dating","first":"initiator","second":"responder",
                     "value":false} ],
    "influenceRules": [
      {"weight":3,"conditions":[{"category":"trait","type":"confident","first":"responder","value":true}]} ],
    "effects": [ {"category":"relationship","type":"dating","first":"initiator","second":"responder",
                  "value":true} ],
    "leadsTo": ["acceptDate", "rejectDate"] }
]}
```

### Example 3 — The runtime loop (the actual API)

This is how a game advances the simulation each turn — the verified Ensemble API:

```js
function step(cast) {
  const volitions = ensemble.calculateVolition(cast);     // score every pair's intents
  const action    = ensemble.getAction("Alice", "Bob",    // best action Alice can do to Bob
                                        volitions, cast);
  if (action) {
    ensemble.doAction(action);          // apply its effects to the social record
    narrate(action);                    // -> "Alice asks Bob out."
  }
  ensemble.setupNextTimeStep();         // advance the clock
  ensemble.runTriggerRules(cast);       // restore derived consistency
}
```

**Tracing Alice → Bob's volition for `askOut`:** romance 70 (> 60 ⇒ **+5**) − shy (**−4**) + Bob confident (influence rule **+3**) = **+4** → Alice *wants* it; if Bob's responder volition doesn't refuse, `getAction` returns `askOut` and `doAction` sets `dating(Alice,Bob)=true`. Nobody scripted that beat — the summed rules produced it. Run this over every pair, every action, every step, and you get *emergent* drama. (`getActions(...)` returns the top-N instead of just one, useful for offering the player a menu.)

## 6. Gotchas & Pitfalls

- **Two rule kinds, two jobs — don't mix them up.** *Trigger rules* are truth-maintenance (no weights, run after every change to keep the record consistent). *Volition rules* are desire (weighted, scored by `calculateVolition`). Putting consistency logic in volition rules — or weights in triggers — produces subtle, hard-to-find bugs.
- **Forgetting `runTriggerRules` after a mutation** leaves the social record contradictory ("dating but also single"). Effects and triggers are separate steps; you must run triggers after `doAction` / any `set`.
- **Volition is *relative*, not absolute.** A +4 action only fires if nothing scores higher *this step*. Tuning is about *relative* weights across the whole set; bumping one weight can silently starve unrelated behaviors.
- **Weight-tuning is finicky and global.** There is no principled scale — authors converge on magic numbers by playtesting, and small changes ripple system-wide.
- **Authoring burden is the real cost.** Believable behavior needs *hundreds* of volition rules and a rich action library; *Prom Week* shipped ~5,000 social rules. Budget for content authoring, not just engine integration.
- **Debugging emergence is brutal.** When a character does something weird, the cause is the *interaction* of dozens of weighted rules, not one line. Use the **Authoring Tool**'s volition explorer (rule-by-rule contribution) — you cannot set a breakpoint on "drama".
- **`directionType` matters.** `directed` (romance A→B ≠ B→A), `undirected` (a personal trait), and `reciprocal` (dating is symmetric) behave differently in `get`/`set` and in rule matching. Picking the wrong one quietly breaks queries.
- **Performance is combinatorial.** `calculateVolition` is ~O(pairs × actions × rules) per step — fine for a classroom cast (~10), needs pruning/caching for large casts.
- **The narrative paradox.** Emergence resists authored arcs; you often *also* need a drama manager / storylets to steer toward intended beats, partly reintroducing the authoring you wanted to avoid.
- **Players read emergence as bugs.** Surface the *reasons* (visible relationship meters, narrated motivations) or a valid-but-surprising event looks like a glitch.
- **It's a JS library, not a service.** No Python binding; integrate it on the JS side (browser/Node/Electron) and treat the notebook layer as conceptual.

## 7. When to Use vs Alternatives

| Approach | Good at | Trade-off vs Ensemble |
| --- | --- | --- |
| **Ensemble** (open-source CiF successor) | Systemic, emergent, **explainable** relationships; deterministic; reusable JS engine + authoring tool | High authoring & debugging cost; weak authorial control; JS-only |
| **Comme il Faut (CiF)** | The original; battle-tested in *Prom Week* | Proprietary C#, not maintained/open — Ensemble is the modern path |
| **Versu / social practices** | Emergent drama via autonomous utility agents + shared *social practices* | Distributed model instead of central scoring; different authoring style, similar costs (see [`social-physics.ipynb`](social-physics.ipynb)) |
| **Dialogue trees / Twine / branching** | Total authorial control; hand-crafted beats | No emergence; branch explosion; brittle to player creativity |
| **Storylets / quality-based narrative** | Middle ground — authored chunks gated by state | Less emergent than rules; scales better than trees |
| **Behavior trees / GOAP / utility AI** | *Individual* task behavior (combat, navigation) | Not relational — no *between*-character state. Often **combined**: utility AI ≈ volition, Ensemble supplies the relational social record. |
| **LLM-driven NPCs** | Open-ended, fluent dialogue; low up-front authoring | Expensive, slow, non-deterministic, hard to constrain/explain. Increasingly **hybridized**: Ensemble for the social state model, an LLM for surface text. |

**Rule of thumb:** reach for Ensemble when *relationships and social consequence are the gameplay*, you want the result explainable and deterministic, and you can invest in rule authoring. Prefer branching/storylets when the *exact story* matters more than emergence; prefer (or hybridize with) an LLM when *fluent open-ended dialogue* matters more than control.

## 8. Resources

**Engine & tooling**
- **Ensemble Engine — source** (JS, open source): https://github.com/ensemble-engine/ensemble
- **Ensemble API wiki** (the methods used above — `loadSocialStructure`, `calculateVolition`, `getAction`, `doAction`, `runTriggerRules`, `set`/`get`): https://github.com/ensemble-engine/ensemble/wiki
- **ensemble-engine org** (authoring tool + related repos): https://github.com/ensemble-engine

**Foundational papers**
- *The Ensemble Engine: Next-Generation Social Physics* (Samuel et al., FDG 2015): http://www.ben-samuel.com/wp-content/uploads/2015/09/FDG2015-The-Ensemble-Engine-Next-Generation-Social-Physics.pdf
- *The Scored Rule Engine: Next-Generation Social Physics* (Samuel, Reed et al. — the generalized rule core): https://www.semanticscholar.org/paper/94b7e4a02b33740d8d589ec726c6dd28219c65f7
- *Prom Week* design retrospective (Road to the IGF): https://www.gamedeveloper.com/design/road-to-the-igf-expressive-intelligence-studio-s-i-prom-week-i-

**Context**
- **Expressive Intelligence Studio** (the lab behind CiF / Prom Week / Ensemble): https://expressiveintelligence.github.io/
- **Ben Samuel — The Ensemble Engine project page**: http://www.ben-samuel.com/projects/the-ensemble-engine/

**Cross-links in this library:** [`social-physics.ipynb`](social-physics.ipynb) (the broad concept & lineage) · [`datalog.ipynb`](datalog.ipynb) / [`swi-prolog.ipynb`](swi-prolog.ipynb) (the predicate/query substrate volition rules are built on) · [`insimul-dsl.ipynb`](insimul-dsl.ipynb) (sibling simulation DSL).